**Make sure you have libraries installed and datasets downloaded**


## Importing libraries

In [19]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


## Preprocessing data and creating train/test/val sets

In [20]:
# Create the train X and y

train_set = pd.read_csv("train.csv")

X = train_set.drop(columns=["target"])

y = train_set["target"]

# Fill missing values

X["text"] = X["text"].fillna("missing")

X["keyword"] = X["keyword"].fillna("missing")

X["location"] = X["location"].fillna("missing")

# TF-IDF settings, ignore words that show more have a frequency aboe 70%

max_df = 0.7

# Vectorize each column separately

text_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=max_df
)

keyword_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=max_df
)

location_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=max_df
)

text_vec = text_vectorizer.fit_transform(X["text"])

keyword_vec = keyword_vectorizer.fit_transform(X["keyword"])

location_vec = location_vectorizer.fit_transform(X["location"])

# Convert to DataFrames to fit back into original X

text_df = pd.DataFrame(
    text_vec.toarray(),
    columns=["text_" + word for word in text_vectorizer.get_feature_names_out()],
    index=X.index
)

keyword_df = pd.DataFrame(
    keyword_vec.toarray(),
    columns=["keyword_" + word for word in keyword_vectorizer.get_feature_names_out()],
    index=X.index
)

location_df = pd.DataFrame(
    location_vec.toarray(),
    columns=["location_" + word for word in location_vectorizer.get_feature_names_out()],
    index=X.index
)

# Remove original text columns and add TF-IDF features

X = pd.concat(
    [
        X.drop(columns=["text", "keyword", "location"]),
        text_df,
        keyword_df,
        location_df
    ],
    axis=1
)

# Picking random number for the random state, make it more random

random_state = 42 

# Create the Test/Train/Val sets

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y, 
    test_size=0.20,
    random_state=random_state
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val, 
    test_size=0.25, 
    random_state=random_state
)


# Create the test X (no y since we are using this to predict a y)

test_dataset = pd.read_csv("test.csv")